<a href="https://colab.research.google.com/github/NaghamZidiah/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully!")

rel = "hf://datasets/FlyRank/internship-warehouse"

Token loaded successfully!
Connected successfully!


In [ ]:
# Inspect the available columns from the warehouse
# using the same table path that worked in ML-05.

sample_query = f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
LIMIT 5
"""

sample_df = con.sql(sample_query).df()

print("Number of columns:", len(sample_df.columns))
print("\nAvailable columns:")
print(sample_df.columns.tolist())

Number of columns: 31

Available columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
signal_check_query = f"""
SELECT
    CASE
        WHEN gsc_impressions = 0 THEN '0'
        WHEN gsc_impressions < 100 THEN '1-99'
        WHEN gsc_impressions < 1000 THEN '100-999'
        ELSE '1000+'
    END AS impression_bucket,

    COUNT(*) AS n,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(gsc_avg_position) AS avg_position

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY 1
ORDER BY
    CASE impression_bucket
        WHEN '0' THEN 1
        WHEN '1-99' THEN 2
        WHEN '100-999' THEN 3
        WHEN '1000+' THEN 4
    END
"""

volume_buckets = con.sql(signal_check_query).df()

print("Volume signal:")
display(volume_buckets)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Volume signal:


,impression_bucket,n,avg_clicks,avg_position
0,0,6230317,0.000000,NaN
1,1-99,2972453,0.057965,16.850414
2,100-999,606189,0.800425,11.017107
3,1000+,32419,5.068787,11.890910


In [ ]:
ctr_position_query = f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,

    COUNT(*) AS n,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(
        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END
    ) AS avg_ctr

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_avg_position IS NOT NULL
  AND gsc_impressions > 0

GROUP BY 1

ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        WHEN '21+' THEN 4
    END
"""

ctr_position_buckets = con.sql(ctr_position_query).df()

print("CTR vs Position signal:")
display(ctr_position_buckets)

CTR vs Position signal:


,position_bucket,n,avg_impressions,avg_clicks,avg_ctr
0,1-3,727362,74.280199,0.282469,0.004756
1,4-10,1456122,94.655608,0.306175,0.003473
2,11-20,519223,56.596118,0.178053,0.002770
3,21+,908354,65.407183,0.085977,0.001289


## 1. My rule and its reason codes

### Signal checks

#### Volume signal

The volume check shows that pages with higher search impressions also have substantially higher average clicks.

**Verdict: CONFIRMED**

This suggests that search volume is a useful signal for prioritizing pages for review. This is an observed relationship in the March 2026 data, not a causal claim.

#### CTR vs Position signal

The CTR vs position check shows that average CTR decreases as average position moves from the top results toward position 21+.

**Verdict: CONFIRMED**

This supports using search position and CTR as signals for identifying pages with weaker search performance. This is an observed relationship in the March 2026 data, not a causal claim.

### My baseline rule

Prioritize pages that have meaningful search demand but weaker search performance. The score will give more priority to pages with higher impressions and weaker average search position.

### Reason codes

- `HIGH_VOLUME` — the page has relatively high search impressions.
- `POSITION_OPPORTUNITY` — the page has a weaker average search position.
- `HIGH_VOLUME_POSITION_OPPORTUNITY` — the page has both relatively high impressions and a weaker average position.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
baseline_query = f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL
"""

baseline_df = con.sql(baseline_query).df()

print("Rows:", len(baseline_df))
print("Columns:", baseline_df.columns.tolist())

display(baseline_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061
Columns: ['report_date', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']


,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,content_a3ea9792f793ec72,11,0,2.272727


In [ ]:
threshold_query = f"""
SELECT
    quantile_cont(gsc_impressions, 0.75) AS impressions_p75,
    quantile_cont(gsc_avg_position, 0.75) AS position_p75,
    quantile_cont(gsc_avg_position, 0.50) AS position_median
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL
"""

thresholds = con.sql(threshold_query).df()

display(thresholds)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impressions_p75,position_p75,position_median
0,62.0,20.2,7.5


In [ ]:
baseline_df["volume_signal"] = (
    baseline_df["gsc_impressions"] >= 62
).astype(int)

baseline_df["position_signal"] = (
    baseline_df["gsc_avg_position"] > 7.5
).astype(int)

baseline_df["score"] = (
    baseline_df["volume_signal"] +
    baseline_df["position_signal"]
)

print("Score distribution:")
display(
    baseline_df["score"]
    .value_counts()
    .sort_index()
)

Score distribution:


,count
score,
0,1269196
1,1982218
2,359647


In [ ]:
# Assign reason codes and actions

baseline_df["reason_code"] = "NO_PRIORITY_SIGNAL"

baseline_df.loc[
    (baseline_df["volume_signal"] == 1) &
    (baseline_df["position_signal"] == 0),
    "reason_code"
] = "HIGH_VOLUME"

baseline_df.loc[
    (baseline_df["volume_signal"] == 0) &
    (baseline_df["position_signal"] == 1),
    "reason_code"
] = "POSITION_OPPORTUNITY"

baseline_df.loc[
    (baseline_df["score"] == 2),
    "reason_code"
] = "HIGH_VOLUME_POSITION_OPPORTUNITY"


baseline_df["action"] = "MONITOR"

baseline_df.loc[
    baseline_df["score"] == 1,
    "action"
] = "REVIEW"

baseline_df.loc[
    baseline_df["score"] == 2,
    "action"
] = "PRIORITIZE_REVIEW"


print("Reason codes:")
display(baseline_df["reason_code"].value_counts())

print("\nActions:")
display(baseline_df["action"].value_counts())

Reason codes:


,count
reason_code,
POSITION_OPPORTUNITY,1436393
NO_PRIORITY_SIGNAL,1269196
HIGH_VOLUME,545825
HIGH_VOLUME_POSITION_OPPORTUNITY,359647



Actions:


,count
action,
REVIEW,1982218
MONITOR,1269196
PRIORITIZE_REVIEW,359647


In [ ]:
# Rank the queue by baseline score

baseline_df = baseline_df.sort_values(
    by=["score", "gsc_impressions", "gsc_avg_position"],
    ascending=[False, False, True]
).reset_index(drop=True)

baseline_df["rank"] = baseline_df.index + 1

print("Top 10 ranked rows:")
display(
    baseline_df[
        [
            "rank",
            "report_date",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

Top 10 ranked rows:


,rank,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action
0,1,2026-03-04,content_945d6ff91386c817,37368,0,8.613948,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
1,2,2026-03-29,content_66288edeb93b7c4f,24577,66,10.794239,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
2,3,2026-03-28,content_66288edeb93b7c4f,23542,165,11.112140,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
3,4,2026-03-28,content_e943d753806d7af3,15522,49,8.788429,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
4,5,2026-03-09,content_e8a52cf3d5988c07,15394,45,17.296804,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
5,6,2026-03-31,content_e6df0936699f5b8f,14682,269,25.035826,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
6,7,2026-03-12,content_e8a52cf3d5988c07,14138,35,16.244306,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
7,8,2026-03-11,content_e8a52cf3d5988c07,13910,30,16.680446,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
8,9,2026-03-10,content_e8a52cf3d5988c07,13060,20,16.536753,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
9,10,2026-03-04,content_e8a52cf3d5988c07,11347,25,16.566053,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW


In [ ]:
# Write the ranked queue to CSV

output_columns = [
    "rank",
    "report_date",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "score",
    "reason_code",
    "action"
]

baseline_queue = baseline_df[output_columns]

output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    output_path,
    index=False
)

print(f"Saved ranked queue to: {output_path}")
print(f"Rows written: {len(baseline_queue)}")
print(f"Columns written: {len(baseline_queue.columns)}")

Saved ranked queue to: work/outputs/baseline_action_score.csv
Rows written: 3611061
Columns written: 9


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-10 Review

I reviewed the top 10 ranked rows to check whether the baseline rule produces reasonable priorities.

The confidence notes are qualitative, not calibrated probabilities. The baseline uses only search impressions and average search position, so the ranking should be treated as a review queue rather than a final decision.

| Rank | Action | Why it's there | Confidence note | What would make it wrong |
|---:|---|---|---|---|
| 1 | PRIORITIZE_REVIEW | High search impressions (37,368) and average position below the median threshold (8.61 > 7.5). | Medium — both baseline signals are present. | The page may already be performing well for its business goal, or the position metric may not represent the page's most important queries. |
| 2 | PRIORITIZE_REVIEW | High search impressions (24,577) and average position below the median threshold (10.79 > 7.5). | Medium — both baseline signals are present. | High impressions may not represent valuable traffic, or the page may have other context that makes review unnecessary. |
| 3 | PRIORITIZE_REVIEW | High search impressions (23,542) and average position below the median threshold (11.11 > 7.5). | Medium — both baseline signals are present. | The page may already have acceptable performance despite the weaker average position. |
| 4 | PRIORITIZE_REVIEW | High search impressions (15,522) and average position below the median threshold (8.79 > 7.5). | Medium — both baseline signals are present. | The observed position may be affected by query mix or other factors not included in the baseline. |
| 5 | PRIORITIZE_REVIEW | High search impressions (15,394) and a relatively weak average position (17.30). | Medium — strong volume and position opportunity signals. | The page may not be a suitable candidate for improvement despite the two signals. |
| 6 | PRIORITIZE_REVIEW | High search impressions (14,682) and a weak average position (25.04). | Medium — both signals strongly support review. | The page may have low-value search intent or other business context that is not represented in this baseline. |
| 7 | PRIORITIZE_REVIEW | High search impressions (14,138) and a weak average position (16.24). | Medium — both baseline signals are present. | The page may already be intentionally targeting lower-ranking queries or have other priorities. |
| 8 | PRIORITIZE_REVIEW | High search impressions (13,910) and a weak average position (16.68). | Medium — both baseline signals are present. | Query-level details or business context could show that the page is not a useful review candidate. |
| 9 | PRIORITIZE_REVIEW | High search impressions (13,060) and a weak average position (16.54). | Medium — both baseline signals are present. | The aggregate position may hide strong performance for the most important queries. |
| 10 | PRIORITIZE_REVIEW | High search impressions (11,347) and a weak average position (16.57). | Medium — both baseline signals are present. | Additional page or business context could make the priority inappropriate. |

## 4. Weak picks + leakage check
Which picks look wrong and why? Confirm no product flags or future windows leaked in.




Some high-ranked rows can still be weak picks because the baseline uses only two signals and does not include query-level or business context. I review a few potentially questionable picks before treating the queue as a decision-support output.

The leakage check confirms that no product flags, label-derived fields, or future-window features are included in the baseline inputs.

In [ ]:
# Review potentially weak picks

weak_picks = baseline_df[
    (baseline_df["score"] == 2) &
    (baseline_df["gsc_clicks"] == 0)
][
    [
        "rank",
        "report_date",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action"
    ]
].head(5)

print("Potentially weak picks:")
display(weak_picks)

Potentially weak picks:


,rank,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action
0,1,2026-03-04,content_945d6ff91386c817,37368,0,8.613948,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
56,57,2026-03-17,content_f6116743b00afc2d,7034,0,9.716520,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
57,58,2026-03-30,content_661a7734f691bef5,7026,0,28.103473,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
59,60,2026-03-19,content_f6116743b00afc2d,6892,0,10.337057,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW
68,69,2026-03-09,content_559cdd76da9306de,6541,0,36.109005,2,HIGH_VOLUME_POSITION_OPPORTUNITY,PRIORITIZE_REVIEW


### Weak-pick interpretation

The weak picks show why the baseline should be treated as a review queue rather than a final decision. These rows have both high impressions and a relatively weak average position, but some have zero clicks. This may indicate a potential opportunity, but it could also reflect query mix, SERP features, or other context that is not captured by this simple baseline.

In [ ]:
# Leakage and excluded-feature check

baseline_input_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

excluded_patterns = [
    "trend",
    "label",
    "target",
    "ai_",
    "product",
    "decision",
    "score",
    "client_hash_id",
    "content_hash_id"
]

violations = []

for feature in baseline_input_features:
    for pattern in excluded_patterns:
        if pattern in feature.lower():
            violations.append((feature, pattern))

print("Baseline input features checked:", len(baseline_input_features))
print("Potential violations:", violations)

assert len(violations) == 0, "Potential leakage/excluded feature detected."

print("Leakage check passed.")

Baseline input features checked: 3
Potential violations: []
Leakage check passed.


### Leakage conclusion

The baseline uses only same-day search performance metrics: impressions, clicks, and average position. No product decision flags, target-derived fields, AI traffic fields, or future-window features are used as baseline inputs.

This check supports the feature design, but it does not guarantee that the baseline is free from every possible source of bias or data-quality issue.

In [ ]:
# Simple clustering baseline for ML-08

import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Use a manageable development sample for clustering
cluster_df = baseline_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].copy()

cluster_df = cluster_df.replace([np.inf, -np.inf], np.nan).dropna()

# Use a reproducible sample for development
cluster_sample = cluster_df.sample(
    n=min(100000, len(cluster_df)),
    random_state=42
)

cluster_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

X_cluster = cluster_sample[cluster_features]

# Standardize features so large-scale variables do not dominate
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Simple 3-group baseline
baseline_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

baseline_labels = baseline_kmeans.fit_predict(X_scaled)

baseline_silhouette = silhouette_score(
    X_scaled,
    baseline_labels
)

cluster_sample["baseline_cluster"] = baseline_labels

print("Clustering baseline sample size:", len(cluster_sample))
print("Number of baseline clusters:", cluster_sample["baseline_cluster"].nunique())
print("Baseline silhouette score:", round(baseline_silhouette, 4))

print("\nCluster sizes:")
display(
    cluster_sample["baseline_cluster"]
    .value_counts()
    .sort_index()
)

print("\nCluster profiles:")
display(
    cluster_sample
    .groupby("baseline_cluster")[cluster_features]
    .mean()
    .round(3)
)

Clustering baseline sample size: 100000
Number of baseline clusters: 3
Baseline silhouette score: 0.6357

Cluster sizes:


,count
baseline_cluster,
0,84800
1,15194
2,6



Cluster profiles:


,gsc_impressions,gsc_clicks,gsc_avg_position
baseline_cluster,,,
0,84.585,0.262,8.700
1,36.662,0.022,55.862
2,23228.000,160.667,1.976


### Clustering baseline results

The clustering baseline produced a silhouette score of 0.6357 on a reproducible development sample of 100,000 rows.

The clusters were highly imbalanced. Cluster 0 contained most of the observations, while Cluster 2 contained only 6 observations. The small cluster appears to capture extreme high-performance observations rather than a broad content archetype.

The baseline provides a useful reference point for ML-08. However, the small cluster should be treated cautiously because it may represent outliers rather than a stable cluster.

This baseline is used only for comparison and is not treated as a ground-truth label or final business decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.